In [9]:
# Test: Check Python version and available libraries
import sys
print("Python version:", sys.version)

# Try importing the libraries we will need
try:
    import numpy as np
    import pandas as pd
    import requests
    import matplotlib.pyplot as plt
    print("✅ All libraries are ready!")
    print("numpy version:", np.__version__)
    print("pandas version:", pd.__version__)
except Exception as e:
    print("❌ Error:", e)

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
✅ All libraries are ready!
numpy version: 2.5.1
pandas version: 3.0.5


In [10]:
import requests
import pandas as pd
import io

def fetch_nasa_power_data(location_name, latitude, longitude, year=2025):
    """
    Fetch hourly Direct Normal Irradiance (DNI) and Temperature (T2M) 
    from NASA POWER API for ANY location in the world.
    """
    print(f"🔄 Fetching data for {location_name} (Lat: {latitude}, Lon: {longitude}) for Year {year}...")
    
    # NASA POWER API Endpoint for hourly data
    url = f"https://power.larc.nasa.gov/api/temporal/hourly/point"
    
    # Parameters to request: DNI (Direct Normal Irradiance) and T2M (Temperature at 2 meters)
    params = {
        "parameters": "ALLSKY_SFC_SW_DNI,T2M",
        "community": "RE", # Renewable Energy community
        "longitude": longitude,
        "latitude": latitude,
        "start": f"{year}0101",
        "end": f"{year}1231",
        "format": "CSV"
    }
    
    response = requests.get(url, params=params, timeout=30)
    
    if response.status_code == 200:
        # NASA API returns metadata header lines before the actual CSV data
        # We find the line that starts with 'YEAR' to start reading the real data
        lines = response.text.splitlines()
        header_idx = next(i for i, line in enumerate(lines) if line.startswith("YEAR"))
        
        csv_data = "\n".join(lines[header_idx:])
        df = pd.read_csv(io.StringIO(csv_data))
        
        # Clean NASA missing values (-999.0)
        df.replace(-999.0, pd.NA, inplace=True)
        df.interpolate(method='linear', inplace=True) # Fill missing gaps linearly
        
        print(f"✅ Successfully loaded {len(df)} hours of data for {location_name}.")
        return df
    else:
        print(f"❌ Error fetching data: {response.status_code}")
        return None 

In [11]:
# تشغيل الدالة لجلب بيانات القاهرة لعام 2023
df_cairo = fetch_nasa_power_data("Cairo, Egypt", 30.04, 31.23, 2025)

# عرض أول 5 صفوف من الجدول عشان نتأكد إن البيانات سليمة
df_cairo.head()

🔄 Fetching data for Cairo, Egypt (Lat: 30.04, Lon: 31.23) for Year 2025...
✅ Successfully loaded 8760 hours of data for Cairo, Egypt.


,YEAR,MO,DY,HR,ALLSKY_SFC_SW_DNI,T2M
0,2025,1,1,0,0.0,8.67
1,2025,1,1,1,0.0,8.29
2,2025,1,1,2,0.0,7.96
3,2025,1,1,3,0.0,7.68
4,2025,1,1,4,0.0,7.50


In [12]:
import matplotlib.pyplot as plt
import pandas as pd

def analyze_solar_resource(df, location_name):
    """
    Analyzes and plots the solar resource for a SINGLE location.
    Generates: 1) Monthly DNI profile, 2) Average Daily DNI profile.
    """
    # 1. Construct proper Datetime index
    df_eda = df.copy()
    df_eda['Datetime'] = pd.to_datetime(df_eda[['YEAR', 'MO', 'DY', 'HR']].rename(
        columns={'YEAR':'year', 'MO':'month', 'DY':'day', 'HR':'hour'}
    ))
    df_eda.set_index('Datetime', inplace=True)

    # 2. Calculate Monthly Aggregation (Convert Wh/m2 to kWh/m2)
    monthly_dni = df_eda.groupby('MO')['ALLSKY_SFC_SW_DNI'].sum() / 1000.0

    # 3. Calculate Average Hourly Profile (To see day/night cycle)
    hourly_dni = df_eda.groupby('HR')['ALLSKY_SFC_SW_DNI'].mean()

    # 4. Setup the Plot (1 row, 2 columns)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot 1: Monthly Profile
    months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    axes[0].bar(monthly_dni.index, monthly_dni.values, color='#e67e22', edgecolor='black')
    axes[0].set_title(f'Monthly Solar Resource (DNI) - {location_name}', fontweight='bold')
    axes[0].set_xlabel('Month')
    axes[0].set_ylabel('Total DNI (kWh/m²/month)')
    axes[0].set_xticks(range(1, 13))
    axes[0].set_xticklabels(months, rotation=45)
    axes[0].grid(axis='y', linestyle='--', alpha=0.7)

    # Plot 2: Average Daily Profile
    axes[1].plot(hourly_dni.index, hourly_dni.values, marker='o', color='#f1c40f', linewidth=2.5)
    axes[1].fill_between(hourly_dni.index, hourly_dni.values, color='#f1c40f', alpha=0.3)
    axes[1].set_title(f'Average Daily DNI Profile - {location_name}', fontweight='bold')
    axes[1].set_xlabel('Hour of the Day (0-23)')
    axes[1].set_ylabel('Average DNI (W/m²)')
    axes[1].set_xticks(range(0, 24, 2))
    axes[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

    # Print annual summary
    total_annual_dni = monthly_dni.sum()
    print(f"📊 Total Annual DNI for {location_name}: {total_annual_dni:,.0f} kWh/m²/year\n")

# --- طريقة استخدام الدالة (بعد ما تسحبي بيانات df_cairo مثلاً) ---
# analyze_solar_resource(df_cairo, "Cairo, Egypt")

In [13]:
def calculate_system_sizing(df, factory_demand_mw=10.0, storage_hours=8.0, solar_multiple=1.5):
    """
    Calculates the required Solar Field Area and TES Capacity for a Solar Tower system.
    """
    # 1. حساب متوسط الإشعاع الشمسي في وقت النهار فقط (عشان نتجنب ساعات الليل اللي الدني فيها صفر)
    daytime_dni = df[df['ALLSKY_SFC_SW_DNI'] > 0]['ALLSKY_SFC_SW_DNI']
    avg_dni = daytime_dni.mean() # الوحدة: واط/متر مربع (W/m²)
    
    # 2. تحويل احتياج المصنع من ميجاوات إلى كيلووات
    demand_kw = factory_demand_mw * 1000
    
    # 3. افتراضات هندسية (كفاءة النظام الشمسي في الأبراج)
    system_efficiency = 0.65 # كفاءة 65% (تشمل فقد المرايا، المستقبل، والأنابيب)
    
    # 4. حساب مساحة المرايا (Solar Field Area)
    # المعادلة: الطلب * معامل التحجيم (Solar Multiple) / (الإشعاع * الكفاءة)
    required_area_m2 = (demand_kw * solar_multiple) / ((avg_dni / 1000) * system_efficiency)
    
    # 5. حساب سعة التخزين الحراري (TES Capacity)
    tes_capacity_mwh = factory_demand_mw * storage_hours
    
    # طباعة التقرير الهندسي
    print(f"📊 التقرير الهندسي للتصميم (بناءً على بيانات الموقع)")
    print("-" * 55)
    print(f"🏭 احتياج المصنع الحراري: {factory_demand_mw} MW_th")
    print(f"☀️ متوسط الإشعاع الشمسي (نهاراً): {avg_dni:,.1f} W/m²")
    print(f"⚙️ معامل التحجيم الشمسي (Solar Multiple): {solar_multiple}")
    print(f"🪞 مساحة المرايا المطلوبة: {required_area_m2:,.0f} متر مربع")
    print(f"🔋 سعة التخزين الحراري (TES): {tes_capacity_mwh:,.0f} ميجاوات/ساعة")
    print("-" * 55 + "\n")
    
    return required_area_m2, tes_capacity_mwh

# --- تجربة الكود على بيانات القاهرة اللي سحبتيها ---
# هنفترض إن عندنا مصنع أسمنت محتاج 10 ميجاوات حرارة، وعايزين تخزين 8 ساعات
area_cairo, tes_cairo = calculate_system_sizing(df_cairo, factory_demand_mw=10.0, storage_hours=8.0)

📊 التقرير الهندسي للتصميم (بناءً على بيانات الموقع)
-------------------------------------------------------
🏭 احتياج المصنع الحراري: 10.0 MW_th
☀️ متوسط الإشعاع الشمسي (نهاراً): 462.3 W/m²
⚙️ معامل التحجيم الشمسي (Solar Multiple): 1.5
🪞 مساحة المرايا المطلوبة: 49,916 متر مربع
🔋 سعة التخزين الحراري (TES): 80 ميجاوات/ساعة
-------------------------------------------------------



In [14]:
def simulate_annual_dispatch(df, mirror_area_m2, tes_capacity_mwh, factory_demand_mw=10.0, system_efficiency=0.65):
    """
    Simulates the hour-by-hour operation of the Solar Tower + TES system over a full year.
    """
    # 1. تجهيز المتغيرات الأساسية
    demand_kw = factory_demand_mw * 1000
    tes_capacity_kwh = tes_capacity_mwh * 1000
    current_tes_charge = 0.0 
    
    total_solar_used_kwh = 0.0
    total_backup_used_kwh = 0.0 
    
    # 2. اللف على كل ساعة في السنة (8760 ساعة)
    for index, row in df.iterrows():
        # حساب الحرارة القادمة من الشمس في الساعة دي
        solar_heat_kw = (row['ALLSKY_SFC_SW_DNI'] / 1000.0) * mirror_area_m2 * system_efficiency
        
        # 3. لو الشمس أكتر من احتياج المصنع (فيه فايض)
        if solar_heat_kw >= demand_kw:
            total_solar_used_kwh += demand_kw # المصنع بياخد احتياجه من الشمس مباشرة
            surplus_kw = solar_heat_kw - demand_kw # الفايض
            
            # شحن الفايض في التخزين (بشرط التخزين ميكونش ملّان)
            available_space = tes_capacity_kwh - current_tes_charge
            charge_amount = min(surplus_kw, available_space)
            current_tes_charge += charge_amount
            
        # 4. لو الشمس أقل من احتياج المصنع (فيه عجز)
        else:
            deficit_kw = demand_kw - solar_heat_kw
            total_solar_used_kwh += solar_heat_kw # بناخد اللي تقدر عليه الشمس
            
            # نسحب العجز من التخزين
            if current_tes_charge >= deficit_kw:
                current_tes_charge -= deficit_kw
                total_solar_used_kwh += deficit_kw 
            else:
                # لو التخزين مكفيش، بنستخدم الغاز الاحتياطي
                total_solar_used_kwh += current_tes_charge
                backup_needed = deficit_kw - current_tes_charge
                total_backup_used_kwh += backup_needed
                current_tes_charge = 0.0 # التخزين فضى

    # 5. حساب النتائج النهائية
    total_demand_kwh = demand_kw * len(df)
    solar_fraction = (total_solar_used_kwh / total_demand_kwh) * 100
    
    print(f"📊 تقرير المحاكاة السنوية (Annual Dispatch)")
    print("-" * 55)
    print(f"🏭 إجمالي احتياج المصنع السنوي: {total_demand_kwh/1e6:,.1f} GWh_th")
    print(f"☀️ إجمالي الحرارة اللي الشمس غطتها: {total_solar_used_kwh/1e6:,.1f} GWh_th")
    print(f"🔥 إجمالي الغاز الاحتياطي المطلوب: {total_backup_used_kwh/1e6:,.1f} GWh_th")
    print(f"📈 نسبة التغطية الشمسية (Solar Fraction): {solar_fraction:.1f}%")
    print("-" * 55 + "\n")
    
    return solar_fraction

# --- تشغيل المحاكاة على بيانات القاهرة ونتايج التصميم اللي طلعت قبل كده ---
# (هنا بنستخدم الـ 49,916 متر والـ 80 ميجاوات اللي طلعوا في الخطوة اللي فاتت)
solar_fraction_cairo = simulate_annual_dispatch(
    df=df_cairo, 
    mirror_area_m2=49916, 
    tes_capacity_mwh=80.0, 
    factory_demand_mw=10.0
)

📊 تقرير المحاكاة السنوية (Annual Dispatch)
-------------------------------------------------------
🏭 إجمالي احتياج المصنع السنوي: 87.6 GWh_th
☀️ إجمالي الحرارة اللي الشمس غطتها: 58.5 GWh_th
🔥 إجمالي الغاز الاحتياطي المطلوب: 29.1 GWh_th
📈 نسبة التغطية الشمسية (Solar Fraction): 66.8%
-------------------------------------------------------



In [15]:
def evaluate_economics(solar_fraction, area_m2, tes_capacity_mwh, factory_demand_mw, material_type="molten_salt"):
    """
    Calculates the Levelized Cost of Heat (LCOH) based on technical sizing,
    financial assumptions, and material properties.
    """
    # 1. الافتراضات المالية (Financial Assumptions)
    project_lifetime = 20       # عمر المشروع بالسنين
    discount_rate = 0.08        # سعر الفايدة / معدل الخصم (8%)
    factory_operating_hours = 8760
    
    # 2. أسعار الوقود الأحفوري (الغاز الطبيعي) كبديل
    gas_price_per_kwh = 0.05    # سعر الغاز (تقريبي 5 سنت لكل كيلوواط/ساعة حرارة)
    
    # 3. أسعار مكونات النظام الشمسي (تقريبية بناءً على أبحاث السوق)
    heliostat_cost_per_m2 = 150 # سعر متر المرايا (دولار)
    bos_and_tower_cost = 2_000_000 # تكاليف البرج والمستقبل والبنية التحتية (دولار)
    
    # 4. خصائص المواد (Material Science Properties)
    materials_db = {
        "molten_salt": {
            "name": "Molten Salt (Nitrate Salts)",
            "cost_per_kwh_th": 25,     # تكلفة سعة التخزين لكل كيلوواط/ساعة
            "opex_ratio": 0.02,        # نسبة الصيانة السنوية من التكلفة (2%)
            "lifetime": 20,            # عمر المادة (سنة)
            "efficiency": 0.95         # كفاءة الشحن والتفريغ
        },
        "refractory_bricks": {
            "name": "Refractory Bricks (High Temp)",
            "cost_per_kwh_th": 40,     # تكلفة أعلى للبدء (CAPEX)
            "opex_ratio": 0.01,        # صيانة أقل (1%) لأنها مش بتتآكل زي الملح
            "lifetime": 30,            # عمر أطول (30 سنة)
            "efficiency": 0.98         # فقد حراري أقل (كفاءة 98%)
        }
    }
    
    mat = materials_db[material_type]
    
    # 5. حساب التكلفة التأسيسية (CAPEX)
    solar_field_cost = area_m2 * heliostat_cost_per_m2
    tes_cost = (tes_capacity_mwh * 1000) * mat["cost_per_kwh_th"]
    total_capex = solar_field_cost + tes_cost + bos_and_tower_cost
    
    # 6. حساب التكاليف التشغيلية السنوية (OPEX)
    annual_opex = total_capex * mat["opex_ratio"]
    
    # 7. حساب تكلفة الغاز المساعد (Backup Fuel Cost)
    demand_kw = factory_demand_mw * 1000
    total_annual_demand_kwh = demand_kw * factory_operating_hours
    # كمية الحرارة اللي مش هتيجي من الشمس هتيجي من الغاز
    gas_heat_needed_kwh = total_annual_demand_kwh * (1 - (solar_fraction / 100))
    annual_backup_cost = gas_heat_needed_kwh * gas_price_per_kwh
    
    # 8. حساب إجمالي التكلفة السنوية (Annualized Cost)
    # معامل استرداد رأس المال (Capital Recovery Factor - CRF) لتوزيع الـ CAPEX على عمر المشروع
    crf = (discount_rate * (1 + discount_rate)**project_lifetime) / ((1 + discount_rate)**project_lifetime - 1)
    annualized_capex = total_capex * crf
    
    total_annual_cost = annualized_capex + annual_opex + annual_backup_cost
    
    # 9. حساب التكلفة المستوية للحرارة (LCOH)
    # إجمالي التكلفة السنوية مقسومة على إجمالي الحرارة اللي الشمس وفرتها
    solar_heat_delivered_kwh = total_annual_demand_kwh * (solar_fraction / 100)
    lcoh = total_annual_cost / solar_heat_delivered_kwh # دولار لكل كيلوواط/ساعة
    
    # طباعة التقرير
    print(f"💰 التقرير الاقتصادي - مادة التخزين: {mat['name']}")
    print("-" * 60)
    print(f"🏗️ التكلفة التأسيسية (CAPEX): ${total_capex:,.0f}")
    print(f"🔧 التكاليف التشغيلية السنوية (OPEX): ${annual_opex:,.0f}")
    print(f"🔥 تكلفة الغاز المساعد سنوياً: ${annual_backup_cost:,.0f}")
    print(f"💵 إجمالي التكلفة السنوية (معدلة): ${total_annual_cost:,.0f}")
    print(f"📉 التكلفة المستوية للحرارة (LCOH): ${lcoh:.4f} / kWh_th")
    print(f"📊 مقارنة: سعر الغاز الحالي هو ${gas_price_per_kwh:.2f} / kWh_th")
    
    if lcoh < gas_price_per_kwh:
        print("✅ قرار استثماري: المشروع الشمسي أوفر من استخدام الغاز!")
    else:
        print("❌ قرار استثماري: الغاز أرخص حالياً (محتاج دعم أو تحسين التصميم).")
    print("-" * 60 + "\n")
    
    return lcoh

# --- تشغيل التقييم الاقتصادي للمادتين ---
print("=== تقييم الخيار الأول: الملح المصهور ===")
lcoh_salt = evaluate_economics(solar_fraction=75, area_m2=49916, tes_capacity_mwh=80, factory_demand_mw=10, material_type="molten_salt")

print("=== تقييم الخيار الثاني: الطوب الحراري ===")
lcoh_bricks = evaluate_economics(solar_fraction=75, area_m2=49916, tes_capacity_mwh=80, factory_demand_mw=10, material_type="refractory_bricks")

=== تقييم الخيار الأول: الملح المصهور ===
💰 التقرير الاقتصادي - مادة التخزين: Molten Salt (Nitrate Salts)
------------------------------------------------------------
🏗️ التكلفة التأسيسية (CAPEX): $11,487,400
🔧 التكاليف التشغيلية السنوية (OPEX): $229,748
🔥 تكلفة الغاز المساعد سنوياً: $1,095,000
💵 إجمالي التكلفة السنوية (معدلة): $2,494,765
📉 التكلفة المستوية للحرارة (LCOH): $0.0380 / kWh_th
📊 مقارنة: سعر الغاز الحالي هو $0.05 / kWh_th
✅ قرار استثماري: المشروع الشمسي أوفر من استخدام الغاز!
------------------------------------------------------------

=== تقييم الخيار الثاني: الطوب الحراري ===
💰 التقرير الاقتصادي - مادة التخزين: Refractory Bricks (High Temp)
------------------------------------------------------------
🏗️ التكلفة التأسيسية (CAPEX): $12,687,400
🔧 التكاليف التشغيلية السنوية (OPEX): $126,874
🔥 تكلفة الغاز المساعد سنوياً: $1,095,000
💵 إجمالي التكلفة السنوية (معدلة): $2,514,114
📉 التكلفة المستوية للحرارة (LCOH): $0.0383 / kWh_th
📊 مقارنة: سعر الغاز الحالي هو $0.05 / kWh_th
✅ قر

In [16]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 1. توليد بيانات تجريبية (Simulating 500 different designs)
np.random.seed(42)
num_samples = 500

# بنجرب مساحات مرايا من 30,000 لـ 80,000 متر
area_samples = np.random.uniform(30000, 80000, num_samples)
# بنجرب ساعات تخزين من 4 لـ 16 ساعة
storage_samples = np.random.uniform(4, 16, num_samples)

# دالة مبسطة لحساب التكلفة التقريبية بناءً على المساحة والتخزين
def quick_cost_estimate(area, storage):
    # تكلفة المرايا والتخزين
    capex = (area * 150) + (storage * 1000 * 25) + 2_000_000
    # افتراض إن الزيادة في المساحة بتزود التغطية الشمسية (بشكل لوغاريتمي عشان التشبع)
    solar_fraction = 0.9 - (0.3 * np.exp(-area / 40000)) - (0.1 * np.exp(-storage / 8))
    gas_cost = 10 * 8760 * (1 - solar_fraction) * 0.05
    return (capex * 0.10 + gas_cost) / (10 * 8760 * solar_fraction) # حساب LCOH تقريبي

# حساب الـ LCOH لكل المحاولات الـ 500
lcoh_samples = [quick_cost_estimate(a, s) for a, s in zip(area_samples, storage_samples)]

# تجهيز البيانات للـ ML
df_ml = pd.DataFrame({
    'Area_m2': area_samples,
    'Storage_hours': storage_samples,
    'LCOH': lcoh_samples
})

# 2. تدريب موديل Machine Learning (Random Forest)
X = df_ml[['Area_m2', 'Storage_hours']]
y = df_ml['LCOH']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ml_model = RandomForestRegressor(n_estimators=100, random_state=42)
ml_model.fit(X_train, y_train)

print("✅ تم تدريب موديل الذكاء الاصطناعي بنجاح!")
print(f"دقة الموديل (R² Score): {ml_model.score(X_test, y_test):.3f}")

# 3. استخدام الـ ML للبحث عن "أفضل تصميم"
# بنخلي الموديل يتوقع التكلفة لـ 1000 احتمال تاني (عشان يلاقي الأفضل)
test_areas = np.random.uniform(30000, 80000, 1000)
test_storages = np.random.uniform(4, 16, 1000)
test_data = pd.DataFrame({'Area_m2': test_areas, 'Storage_hours': test_storages})

# الموديل بيتنبأ بالتكلفة
predicted_lcoh = ml_model.predict(test_data)

# بنلاقي أقل تكلفة (أفضل تصميم)
best_idx = np.argmin(predicted_lcoh)
best_area = test_areas[best_idx]
best_storage = test_storages[best_idx]
best_lcoh = predicted_lcoh[best_idx]

print("\n" + "="*50)
print("🤖 توصية الذكاء الاصطناعي (AI Optimization):")
print("="*50)
print(f"🪞 أفضل مساحة مرايا: {best_area:,.0f} متر مربع")
print(f"🔋 أفضل ساعات تخزين: {best_storage:,.1f} ساعة")
print(f"📉 أقل تكلفة متوقعة (LCOH): ${best_lcoh:.4f} / kWh_th")
print("="*50)

✅ تم تدريب موديل الذكاء الاصطناعي بنجاح!
دقة الموديل (R² Score): 0.999

🤖 توصية الذكاء الاصطناعي (AI Optimization):
🪞 أفضل مساحة مرايا: 30,581 متر مربع
🔋 أفضل ساعات تخزين: 11.3 ساعة
📉 أقل تكلفة متوقعة (LCOH): $10.6884 / kWh_th
